If you are planning to use the **Parameterized Model Memory Analyzer for Microsoft Fabric semantic models**, then execute the following steps:
1. Download the wheel from https://github.com/tarente/FabricDataEngineering/blob/main/param_model_memory_analyzer-0.1.1-py3-none-any.whl to you local machine.
1. Open Explorer -> Resources -> Built-in, and select ...
1. Upload files and select the download wheel.

The above steps just need to be executed once before running the Notebook.


In [ ]:
%pip install semantic-link-labs --q


If you are planning to **not** use the **Parameterized Model Memory Analyzer for Microsoft Fabric semantic models**, then you need do change the line 5 to:

mma_version = **"sempy"** # "pmma" for the customized library, any other option will use sempy

Otherwise, there is not changes needed.




In [ ]:
# Choose to use the sempy or simplied version pmma (param_model_memory_analyzer)
mma_version = "pmma" # "pmma" for the customized library, any other option will use sempy


In [ ]:
# Import the choosen library
if mma_version == "pmma":
    import subprocess

    # Load the wheel only if needed
    subprocess.check_call([
        "pip", "install",
        "./builtin/param_model_memory_analyzer-0.1.1-py3-none-any.whl",
        "-q"
    ])

    import param_model_memory_analyzer as pmma
else:
    import sempy.fabric as fabric


In [ ]:
# Imports
from sempy_labs import admin
import pandas as pd


The cell below 👇🏻 may be hidden, but you can unhide it to view the code.

In [ ]:
# Define the get_ppu_storage function
if mma_version == "pmma":
    def get_ppu_storage():
        return pmma.get_ppu_storage()
else:
    def get_ppu_storage():
        # Get the PPU (PP3) Capacity
        capacity_id = (
            admin.list_capacities()
                .loc[lambda pd: pd["Sku"].eq("PP3"), "Capacity Id"]
                .iloc[0]
        )

        # Get the list of Workspaces assigned to PPU
        pd_workspaces = (
            admin.list_workspaces()
                .loc[lambda pd: pd["Capacity Id"].eq(capacity_id)]
                .rename(columns={"Id": "Workspace Id", "Name": "Workspace Name"})
                .loc[:, ["Workspace Id", "Workspace Name"]]
        )

        # Get the list of Semantic Models (Datasets) in the PPU Workspaces
        pd_list_datasets = (
            admin.list_datasets()
                .merge(pd_workspaces, on="Workspace Id", how="left")
                .loc[lambda pd: ~pd["Content Provider Type"].isin(["Unknown"])]
                .sort_values(["Workspace Name", "Dataset Name"])
        )

        # Iterate over all Semantic Models (Datasets) in the PPU Workspaces and calculate their size
        list_dataset_info = []

        for r in pd_list_datasets.to_dict("records"):
            workspace_name = r["Workspace Name"]
            workspace_id   = r["Workspace Id"]
            dataset_name   = r["Dataset Name"]
            dataset_id     = r["Dataset Id"]

            try:
                # --- HARD SANDBOX: catch ANY exception, including XMLA session failures ---
                try:
                    pd_analysis = fabric.model_memory_analyzer(
                        dataset          = dataset_id,
                        workspace        = workspace_id,
                        return_dataframe = True
                    )

                except Exception as inner_exc:
                    raise RuntimeError(f"Analyzer failed: {inner_exc}")

                # Extract Total Size
                total_size = pd_analysis["Model Summary"]["Total Size"].iloc[0] or 0

                list_dataset_info.append(
                    {
                        "workspace_name": workspace_name,
                        "workspace_id":   workspace_id,
                        "dataset_name":   dataset_name,
                        "dataset_id":     dataset_id,
                        "total_size":     int(total_size)
                    }
                )

            except Exception as exc:
                error_message = str(exc)
                date_utc_index = error_message.find("\nDate (UTC): ")
                if date_utc_index == -1:
                    date_utc_index = error_message.find("Date (UTC): ")
                if date_utc_index != -1:
                    end_of_line = error_message.find("\n", date_utc_index + 1)
                    if end_of_line == -1:
                        end_of_line = len(error_message)
                    error_message = error_message[:end_of_line].rstrip()

                print(
                    f"❌ Error analyzing 'Workspace Name' = '{workspace_name}' ('{workspace_id}'), "
                    f"'Dataset Name' = '{dataset_name}' ('{dataset_id}'): '{error_message}'"
                )

                list_dataset_info.append(
                    {
                        "workspace_name": workspace_name,
                        "workspace_id":   workspace_id,
                        "dataset_name":   dataset_name,
                        "dataset_id":     dataset_id,
                        "total_size":     0
                    }
                )

        return pd.DataFrame(list_dataset_info)


In [ ]:
# Get the Semantic Models (Datasets) in the PPU Workspaces and calculate their size
pd_dataset_info = (
    get_ppu_storage()
)


In [ ]:
# List the size of all Semantic Models (Datasets) in the PPU Workspaces
pd_dataset_sizes = (
    pd_dataset_info
        .rename(
            columns={
                "workspace_name": "Workspace Name",
                "workspace_id":   "Workspace Id",
                "dataset_name":   "Dataset Name",
                "dataset_id":     "Dataset Id",
                "total_size":     "Total Size (bytes)"
            }
        )
)


In [ ]:
# Aggregate the size of all Semantic Models (Datasets) per Workspace and total at Tenant level
display(
    pd_dataset_sizes
        .agg({"Total Size (bytes)": "sum"})
        .to_frame()
        .T
        .assign(
            Workspace_Name="PPU at Tenant",
            Total_Size_MB=lambda df: (df.iloc[:, 0] / 1024.0**2).round(2),
            Total_Size_GB=lambda df: (df.iloc[:, 0] / 1024.0**3).round(2),
        )
        [["Workspace_Name", "Total_Size_MB", "Total_Size_GB"]]
)
